In [ ]:
import os, glob
import cv2
import numpy as np
import plotly.express as px

import nibabel
from nibabel import orientations
import dicom2nifti
import pydicom
from dicom2nifti.common import sort_dicoms
from dicom2nifti.exceptions import ConversionError

def create_affine_bis(sorted_dicoms):
    """
    Function to generate the affine matrix for a dicom series
    This method was based on (http://nipy.org/nibabel/dicom/dicom_orientation.html)

    :param sorted_dicoms: list with sorted dicom files
    """

    # Create affine matrix (http://nipy.sourceforge.net/nibabel/dicom/dicom_orientation.html#dicom-slice-affine)
    image_orient1 = np.array(sorted_dicoms[0].ImageOrientationPatient)[0:3]
    image_orient2 = np.array(sorted_dicoms[0].ImageOrientationPatient)[3:6]

    delta_r = float(sorted_dicoms[0].PixelSpacing[0])
    delta_c = float(sorted_dicoms[0].PixelSpacing[1])

    image_pos = np.array(sorted_dicoms[0].ImagePositionPatient)

    last_image_pos = np.array(sorted_dicoms[-1].ImagePositionPatient)

    if len(sorted_dicoms) == 1:
        # Single slice
        slice_thickness = 1
        if "SliceThickness" in sorted_dicoms[0]:
            slice_thickness = sorted_dicoms[0].SliceThickness
        step = - np.cross(image_orient1, image_orient2) * slice_thickness
    else:
        step = (image_pos - last_image_pos) / (1 - len(sorted_dicoms))

    # check if this is actually a volume and not all slices on the same location
    if np.linalg.norm(step) == 0.0:
        raise ConversionError("NOT_A_VOLUME")

    affine = np.array(
        [[image_orient1[0] * delta_r, image_orient2[0] * delta_c, -step[0], image_pos[0]],
         [image_orient1[1] * delta_r, image_orient2[1] * delta_c, -step[1], image_pos[1]],
         [image_orient1[2] * delta_r, image_orient2[2] * delta_c, step[2], image_pos[2]],
         [0, 0, 0, 1]]
    )
    return affine

def display_st(rt, nii):
    dcm = pydicom.dcmread(rt)
    roi_names = {roi.ROINumber: roi.ROIName for roi in dcm.StructureSetROISequence}
    original_ctr = []
    for roi_contour in dcm.ROIContourSequence:
        if roi_names[roi_contour.ReferencedROINumber] == structure_name:
            for contour in roi_contour.ContourSequence:
                coords = contour.ContourData
                # (x0, y0, z0, x1, y1, z1, ...)
                points = list(zip(coords[0::3], coords[1::3], coords[2::3]))
                original_ctr.extend(points)
            break
    
    original_ctr = np.asarray(original_ctr, dtype="int64")
    voxel_coords = convert_ctr_to_voxel_space(original_ctr, nii.affine)
    original_mask = fill_vol_ctrs(nii.dataobj.shape, voxel_coords)
    fig = px.imshow(np.moveaxis(original_mask, -1, 0), 
                    animation_frame=0, 
                    binary_string=True, 
                    labels=dict(animation_frame="slice"))
    fig.show()

def print_ort_affine_rt(nii, rt):
    current_ornt = orientations.io_orientation(nii.affine)
    current_ornt = orientations.ornt2axcodes(current_ornt)
    print(current_ornt)
    print(nii.affine)
    fig = px.imshow(np.moveaxis(nii.dataobj, -1, 0), 
                    animation_frame=0, 
                    binary_string=True, 
                    labels=dict(animation_frame="slice"))
    fig.show()
    display_st(rt, nii)

def convert_ctr_to_voxel_space(original_ctr, affine):
    inv_affine = np.linalg.inv(affine)
    
    # Add 1 for homogeneous coordinates
    original_ctr = np.hstack((original_ctr, np.ones((original_ctr.shape[0], 1))))
    
    voxel_coords = inv_affine @ original_ctr.transpose()
    
    voxel_coords = voxel_coords.transpose().astype("int64")
    
    # Drop the homogeneous coordinate
    voxel_coords = voxel_coords[:,:3]
    
    return voxel_coords

def fill_vol_ctrs(shape, ctrs):
    """
    ctrs must be like (x,y,z)
    """
    mask = np.zeros(shape, dtype="uint8")
    for z in np.unique(ctrs[:,2]):
        img = np.zeros(shape[:2], dtype="uint8")
        zctrs = ctrs[ctrs[:,2] == z][:,:2]
        cv2.fillPoly(img, [zctrs], 1)
        mask[:,:,z] = img
    return mask

TEMP_FILE_SAVE_NII = r"c:\Users\bguet\Desktop\work\tmp\nii"

ct = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"
rt = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_RTst_2017-06-21_100509_ORL.(sauf.sinus)_CT.0.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402100.462.1358.dcm"

structure_name = "Contour-exte"


# load without no change
nii = dicom2nifti.convert_dicom.dicom_series_to_nifti(
            original_dicom_directory=ct, 
            output_file=None,
            reorient_nifti=False)["NII"]
print_ort_affine_rt(nii, rt)

# load with nifti defualt reorientation
nii = dicom2nifti.convert_dicom.dicom_series_to_nifti(
            original_dicom_directory=ct, 
            output_file=TEMP_FILE_SAVE_NII,
            reorient_nifti=True)["NII"]
print_ort_affine_rt(nii, rt)

# load and create affine bis
nii = dicom2nifti.convert_dicom.dicom_series_to_nifti(
            original_dicom_directory=ct, 
            output_file=None,
            reorient_nifti=False)["NII"]
sorted = sort_dicoms(list(map(pydicom.dcmread, glob.glob(os.path.join(ct, "*.dcm")))))
nii = nibabel.Nifti1Image(nii.dataobj, create_affine_bis(sorted))
print_ort_affine_rt(nii, rt)

# load and reorient into RAS
nii = dicom2nifti.convert_dicom.dicom_series_to_nifti(
            original_dicom_directory=ct, 
            output_file=None,
            reorient_nifti=False)["NII"]
current_ornt = orientations.io_orientation(nii.affine)
target_ornt = orientations.axcodes2ornt(('R', 'A', 'S'))
transform = orientations.ornt_transform(current_ornt, target_ornt)
new_data = orientations.apply_orientation(nii.dataobj, transform)
new_affine = nii.affine @ orientations.inv_ornt_aff(transform, nii.dataobj.shape)
nii = nibabel.Nifti1Image(new_data, new_affine)
print_ort_affine_rt(nii, rt)

# load and reorient into PLS and create affine bis
nii = dicom2nifti.convert_dicom.dicom_series_to_nifti(
            original_dicom_directory=ct, 
            output_file=None,
            reorient_nifti=False)["NII"]
current_ornt = orientations.io_orientation(nii.affine)
target_ornt = orientations.axcodes2ornt(('P', 'L', 'S'))
transform = orientations.ornt_transform(current_ornt, target_ornt)
new_data = orientations.apply_orientation(nii.dataobj, transform)
sorted = sort_dicoms(list(map(pydicom.dcmread, glob.glob(os.path.join(ct, "*.dcm")))))
nii = nibabel.Nifti1Image(new_data, create_affine_bis(sorted))
print_ort_affine_rt(nii, rt)

# Total Segmentator

"head_glands_cavities": {
        1: "eye_left",
        2: "eye_right",
        3: "eye_lens_left",
        4: "eye_lens_right",
        5: "optic_nerve_left",
        6: "optic_nerve_right",
        7: "parotid_gland_left",
        8: "parotid_gland_right",
        9: "submandibular_gland_right",
        10: "submandibular_gland_left",
        11: "nasopharynx",
        12: "oropharynx",
        13: "hypopharynx",
        14: "nasal_cavity_right",
        15: "nasal_cavity_left",
        16: "auditory_canal_right",
        17: "auditory_canal_left",
        18: "soft_palate",
        19: "hard_palate"
    }

In [ ]:
from PIL import Image
import numpy as np
import plotly.express as px
import dicom_class

path = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"

img = dicom_class.CT(path)

fig = px.imshow(np.moveaxis(img.get_voxel_array(), -1, 0), 
                animation_frame=0, 
                binary_string=True, 
                labels=dict(animation_frame="slice"))
fig.show()

Image.fromarray(np.moveaxis(img.get_voxel_array(), -1, 0)[0]).show()

In [12]:
TEMP_FILE_SAVE_NII = r"c:\Users\bguet\Desktop\work\tmp\nii"

In [ ]:
path = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"

In [ ]:
from totalsegmentator.python_api import totalsegmentator
import dicom_class

ct = dicom_class.CT(path)
ct.load_nii()
nii = ct.nii
output_img = totalsegmentator(nii, task="head_glands_cavities")

In [ ]:
import numpy as np

out = output_img.dataobj
np.min(out), np.max(out)

In [ ]:
import numpy as np

print(np.shape(out))

In [ ]:
import plotly.express as px

fig = px.imshow(np.moveaxis(out, -1, 0), 
                animation_frame=0, 
                binary_string=True, 
                labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
totalsegmentator_cls_idx = {
        1: "eye_left",
        2: "eye_right",
        3: "eye_lens_left",
        4: "eye_lens_right",
        5: "optic_nerve_left",
        6: "optic_nerve_right",
        7: "parotid_gland_left",
        8: "parotid_gland_right",
        9: "submandibular_gland_right",
        10: "submandibular_gland_left",
        11: "nasopharynx",
        12: "oropharynx",
        13: "hypopharynx",
        14: "nasal_cavity_right",
        15: "nasal_cavity_left",
        16: "auditory_canal_right",
        17: "auditory_canal_left",
        18: "soft_palate",
        19: "hard_palate"
    }

In [ ]:
import plotly.express as px
import cv2

pl_mask = (out == 7).astype("uint8")
print(pl_mask.shape)
contours = []
for z, mask in enumerate(np.moveaxis(pl_mask, -1, 0)):
    ctrs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ctrs = [(*xy, z) for xy in np.squeeze(ctrs)]
    contours.extend(ctrs)

print(np.shape(contours))

contours = np.array(contours)
fig = px.scatter_3d(x=contours[:,0], y=contours[:,1], z=contours[:,2])
fig.update_layout(
    scene = dict(
        xaxis = dict(range=[0, 512],),
        yaxis = dict(range=[0, 512],),
        zaxis = dict(range=[0, 198],),))
fig.show()

In [3]:
import cv2
import numpy as np
import dicom_class

def fill_vol_ctrs(shape, ctrs):
    """
    ctrs must be like (x,y,z)
    """
    mask = np.zeros(shape, dtype="uint8")
    for z in np.unique(ctrs[:,2]):
        img = np.zeros(shape[:2], dtype="uint8")
        zctrs = ctrs[ctrs[:,2] == z][:,:2]
        cv2.fillPoly(img, [zctrs], 1)
        mask[:,:,z] = img
    return mask


structure_ID = 7
structure_name = "Parotide_homolaterale"
# struct_mask = (out == structure_ID).astype("uint8")

# rt = dicom_class.RTSTRUCT(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_RTst_2017-06-21_100509_ORL.(sauf.sinus)_CT.0.DOSI.0_n1__00000\2.16.840.1.114362.1.11956109.23961857828.605402100.462.1358.dcm")
# ct = dicom_class.CT(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000")
# ct.add_rtstruct(rt)
# voxel_coords = rt.get_contours(structure_name)
# original_mask = fill_vol_ctrs(struct_mask.shape, voxel_coords)

# dice = (np.logical_and(struct_mask, original_mask).sum()) / (np.logical_or(struct_mask, original_mask).sum())
# print(dice)

In [ ]:
img = nii.dataobj

fig = px.imshow(np.moveaxis(img, -1, 0), 
                animation_frame=0, 
                binary_string=True, 
                labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
import plotly.express as px

fig = px.imshow(np.moveaxis(struct_mask, -1, 0), 
                animation_frame=0, 
                binary_string=True, 
                labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
import plotly.express as px

fig = px.imshow(np.moveaxis(original_mask, -1, 0), 
                animation_frame=0, 
                binary_string=True, 
                labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
import nibabel
import os, glob
import cv2
from nibabel import orientations

import numpy as np
import dicom2nifti
import plotly.express as px
import pydicom
from dicom2nifti.common import sort_dicoms
from dicom2nifti.exceptions import ConversionError

structure_name = "Contour-exte"


def create_affine_bis(sorted_dicoms):
    """
    Function to generate the affine matrix for a dicom series
    This method was based on (http://nipy.org/nibabel/dicom/dicom_orientation.html)

    :param sorted_dicoms: list with sorted dicom files
    """

    # Create affine matrix (http://nipy.sourceforge.net/nibabel/dicom/dicom_orientation.html#dicom-slice-affine)
    image_orient1 = np.array(sorted_dicoms[0].ImageOrientationPatient)[0:3]
    image_orient2 = np.array(sorted_dicoms[0].ImageOrientationPatient)[3:6]

    delta_r = float(sorted_dicoms[0].PixelSpacing[0])
    delta_c = float(sorted_dicoms[0].PixelSpacing[1])

    image_pos = np.array(sorted_dicoms[0].ImagePositionPatient)

    last_image_pos = np.array(sorted_dicoms[-1].ImagePositionPatient)

    if len(sorted_dicoms) == 1:
        # Single slice
        slice_thickness = 1
        if "SliceThickness" in sorted_dicoms[0]:
            slice_thickness = sorted_dicoms[0].SliceThickness
        step = - np.cross(image_orient1, image_orient2) * slice_thickness
    else:
        step = (image_pos - last_image_pos) / (1 - len(sorted_dicoms))

    # check if this is actually a volume and not all slices on the same location
    if np.linalg.norm(step) == 0.0:
        raise ConversionError("NOT_A_VOLUME")

    affine = np.array(
        [[image_orient1[0] * delta_r, image_orient2[0] * delta_c, -step[0], image_pos[0]],
         [image_orient1[1] * delta_r, image_orient2[1] * delta_c, -step[1], image_pos[1]],
         [image_orient1[2] * delta_r, image_orient2[2] * delta_c, step[2], image_pos[2]],
         [0, 0, 0, 1]]
    )
    return affine


# load and reorient into PLS and create affine bis
nii = dicom2nifti.convert_dicom.dicom_series_to_nifti(
            original_dicom_directory=ct, 
            output_file=None,
            reorient_nifti=False)["NII"]
current_ornt = orientations.io_orientation(nii.affine)
target_ornt = orientations.axcodes2ornt(('P', 'L', 'S'))
transform = orientations.ornt_transform(current_ornt, target_ornt)
new_data = orientations.apply_orientation(nii.dataobj, transform)
sorted = sort_dicoms(list(map(pydicom.dcmread, glob.glob(os.path.join(ct, "*.dcm")))))
nii = nibabel.Nifti1Image(new_data, create_affine_bis(sorted))
print_ort_affine_rt(nii, rt)




calcul of affine matrix seems wrong as first column is multipled by delta_c and second by delta_r
it should be the opposite with the signs inverted for first two rows

In [6]:
import pydicom

dcm = pydicom.dcmread(rt)

for roi in dcm.StructureSetROISequence:
    print(roi.ROIName)

ISO
coordonnees table
ATM_controlaterale
ATM_homolaterale
Bouche_oesophagienne
Canal_medullaire
Encephale
Constricteur_inf
LARYNX
LEVRES
Mandibule
Constricteur_med
Oesophage
Oreille_Int_controlaterale
Oreille_Int_homolaterale
Parotide_controlaterale
Parotide_homolaterale
Sous_max_controlateral
Constricteur_sup
Tronc_cerebral
Sous_max_homolateral
Cavite_Buccale
CTV_70Gy_(2Gy)
CTV_63Gy_(1,8Gy)
CTV_56Gy_(1,6Gy)
C_ext
arc dentaire
densite 1
C_ext-3mm
PTV_70Gy_(2Gy)
PTV_63Gy_(1,8Gy)
PTV_56Gy_(1,6Gy)
Moelle+5mm
Tronc_cerebral+5mm
Cavite_buccale_horsPTV
Larynx_horsPTV
Mandibule_horsPTV
PTV_dosi_70Gy
Ring_PTV_70
PTV_dosi_63Gy
Ring_PTV_63
PTV_dosi_56Gy
Ring_PTV_56
Tissus_Sains_1
Tissus_Sains_2
Tissus_Sains_3
Overlap_Parotide_HL
Overlap_Parotide_CL
Cext--calcul
tableE-int
tableE-ext
Contour-exte
Région d'intérêt externe
MUSCLE CONSTRICTEUR
